# Bonus: API Enrichment: Worked Example
**DATA 271 Midterm 2 Take-Home | Spring 2026**

---

Your dataset already has useful information in it — but often there's related data living somewhere else on the web that could make your analysis richer. **The Bonus asks you to reach out to an external API, pull in at least one piece of information, and add it back to your DataFrame as a new column.**

This is the same core pattern we used in lecture with the YouTube API:

```
requests.get(url)  →  .json()  →  extract what you need  →  map back to your DataFrame
```

The worked example below uses the **Wikipedia REST API** which is handy because you do not need an API key! But scroll to the bottom for other APIs that may be a better fit for your specific dataset.

---

## 🌐 The Wikipedia REST API

Wikipedia exposes a free public API that returns a plain-text summary for any topic. It's perfect for adding a description column to any dataset that has a categorical column with named things — cancer types, countries, sports, companies, drugs, etc.

**Endpoint:**
```
https://en.wikipedia.org/api/rest_v1/page/summary/{TERM}
```

**You do not need an API key.**

All you need is to `import requests` :)

---
## Step 1 — Test the API on a single term

In [ ]:
import requests
import pandas as pd

# Try one term manually first — always test before looping!
term = "Melanoma"
url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{term.replace(' ', '_')}"

response = requests.get(url)
print("Status code:", response.status_code)  # 200 = success

data = response.json()
data

In [ ]:
# The field we want is 'extract' — the plain-English summary
print(data['extract'])

---
## Step 2 — Wrap it in a function

In [ ]:
def get_wiki_summary(term):
    """
    Calls the Wikipedia REST API and returns a one-sentence
    plain-text summary for the given term.
    Returns None if the page isn't found (status != 200).
    """
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{term.replace(' ', '_')}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        return data.get('extract', None)
    else:
        return None

# Quick test
get_wiki_summary("Melanoma")

---
## Step 3 — Call the API for each unique value in your column

Always loop over **unique values** — not every row. If you have 500 rows but only 12 unique cancer types, you only need 12 API calls.

In [ ]:
# -------------------------------------------------------------------
# Replace 'cancer_type' below with the column name from YOUR dataset
# -------------------------------------------------------------------

# Example dataset — swap this out for your own df
df = pd.DataFrame({
    'cancer_type': ['Melanoma', 'Leukemia', 'Melanoma', 'Lung cancer',
                    'Leukemia', 'Breast cancer', 'Lung cancer', 'Melanoma'],
    'survival_rate': [0.93, 0.68, 0.91, 0.25, 0.72, 0.91, 0.28, 0.89]
})

unique_terms = df['cancer_type'].dropna().unique()
print(f"Unique terms to look up: {unique_terms}")

In [ ]:
# Call the API once per unique term and store in a dict
summaries = {}

for term in unique_terms:
    summaries[term] = get_wiki_summary(term)
    print(f"✓ {term}")  # progress indicator

summaries

---
## Step 4 — Map the results back into your DataFrame

In [ ]:
# .map() matches each row's value to the dict key and fills in the result
df['wiki_summary'] = df['cancer_type'].map(summaries)

df[['cancer_type', 'wiki_summary']]

---
## ✅ What to include in your submission

Show:
1. The API call working (a test on one term, Step 1)
2. The loop over your unique values (Step 3)
3. A `df.head()` showing the new column in your actual dataset (Step 4)

Then write **1–2 sentences** answering: *What did you add, and how could this new column be useful in your final project analysis?*

In [ ]:
# Your written response here (as a comment or in the Markdown cell below):
# I added a 'wiki_summary' column that ...

*(type your 1–2 sentence interpretation here)*

---
## 🔄 Adapting this to your dataset

You don't have to use Wikipedia. Pick whichever API is most useful for your project. The pattern is always the same — only the URL and the field you extract will change.

| Your dataset | Suggested API | What to add | Key field to extract |
|---|---|---|---|
| Cancer (Kaggle) | Wikipedia | Disease description | `'extract'` |
| Cancer (Kaggle) | OpenFDA | Drug class or approval info | `'drug_interactions'`, `'indications_and_usage'` |
| CA Census | Census Bureau API | Additional demographic variable (income, unemployment) | Varies by query |
| CA Census | CA Open Data (`data.ca.gov`) | County-level stats | Varies |
| Sports / gambling | TheSportsDB | Team league, sport, country | `'strLeague'`, `'strCountry'` |
| Sports / gambling | Wikipedia | Team or sport description | `'extract'` |
| Any dataset with countries | REST Countries | Region, population, currency | `'region'`, `'population'` |
| Any dataset with dates + location | Open-Meteo | Historical weather (temp, precipitation) | `'temperature_2m_mean'` |

---

### Alternative API examples (no key required)

**REST Countries** — for datasets with a country column:
```python
url = f"https://restcountries.com/v3.1/name/{country_name}"
data = requests.get(url).json()
region = data[0]['region']   # e.g. 'Americas'
```

**OpenFDA** — for drug or condition names:
```python
url = f"https://api.fda.gov/drug/label.json?search=openfda.brand_name:{drug_name}&limit=1"
data = requests.get(url).json()
# extract from data['results'][0] ...
```

**TheSportsDB** — for team names (free tier, no key):
```python
url = f"https://www.thesportsdb.com/api/v1/json/3/searchteams.php?t={team_name}"
data = requests.get(url).json()
league = data['teams'][0]['strLeague']
```